In [1]:
from datetime import datetime
from utils_CI_analysis import *
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, IntSlider
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd

cmap = LinearSegmentedColormap.from_list('black_white_black', ['black', 'white', 'black'], N=256)

start_date = datetime(2022, 1, 1, 0, 0)
end_date = datetime(2022, 12, 31, 23, 59, 59)

In [2]:
import ipympl

In [3]:
# data preparation:
_dfs = load_data()

# Ranking analysis

In [5]:
# Converting dictionary of dataframe into a single dataframe
_tmp = pd.concat(
    [
        _dfs[country][["datetime", "CI_direct"]]
        .set_index("datetime")
        .rename(columns={"CI_direct": country})
        for country in _dfs.keys()
    ],
    axis=1,
    sort=False,
)

# Ranking each row based on the value
_tmp = _tmp.rank(ascending=True, method="first", axis=1)

# Converting rank dataframe into tally table with rank as the index
_tmp = pd.concat(
    [
        _tmp[col]
        .value_counts()
        .reset_index()
        .rename(columns={col: "rank", "count": col})
        .set_index("rank")
        for col in _tmp.columns
    ],
    axis=1,
    sort=True,
).fillna(0.0)

In [6]:
# Ordering countries based on the count value for each rank
order_col = []
for row in _tmp.iterrows():
       # Sorting the values only when they are non-zero 
       order_col += row[1][row[1] != 0].drop(index=order_col, errors='ignore').sort_values(ascending=False).index.to_list()

In [7]:
_tmp.loc[:,order_col].plot(kind='barh', stacked=True, title="Rank at hourly level")

<Axes: title={'center': 'Rank at hourly level'}, ylabel='rank'>

# Cross correlation analysis

In [11]:
countries = _dfs.keys()

In [12]:
countries

dict_keys(['Germany', 'Ireland', 'Great Britain', 'France', 'Sweden', 'Finland', 'Belgium', 'Brazil', 'Denmark', 'Estonia', 'Spain', 'Hungary', 'Singapore', 'Italy', 'Japan', 'South Africa', 'Uruguay', 'Croatia'])

In [13]:
god_df = pd.concat(
    [
        _dfs[country][["datetime", "CI_direct"]]
        .set_index("datetime")
        .rename(columns={"CI_direct": country})
        for country in _dfs.keys()
    ],
    axis=1,
    sort=False,
)[
    ["Belgium",
        "Germany",
        "France",
        "Spain",
        "Italy",
        "Great Britain",
        "Ireland",
        "Denmark",
        "Sweden",
        "Finland",
        "Estonia",
        "Hungary",
        "Croatia",
        "Uruguay",
        "Brazil",
        "Singapore",
        "Japan",
        "South Africa",    
    ]
]

In [14]:
# Interactive function for heatmap with a sliding window
def interactive_heatmap(start_row=0, window_size=100):
    # Ensure the window size does not exceed the DataFrame length
    end_row = start_row + window_size
    end_row = min(end_row, len(god_df))
    
    # Calculate correlation on the sliding window
    corr_matrix = god_df.iloc[start_row:end_row].corr()

    # Plot heatmap
    plt.figure(figsize=(15, 15))
    sns.heatmap(corr_matrix, annot=True, cmap=cmap, vmin=-1.0, vmax=1.0)
    plt.title(f'Correlation Matrix from Row {start_row} to {end_row}')
    plt.show()

# Create slider widgets
start_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(god_df) - 1,
    step=1,
    description='Start Row:',
    continuous_update=False
)

window_slider = widgets.IntSlider(
    value=100,
    min=10,
    max=len(god_df),
    step=10,
    description='Window Size:',
    continuous_update=False
)

# Display interactive widget
interact(interactive_heatmap, start_row=start_slider, window_size=window_slider)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Start Row:', max=8759), IntSlid…

<function __main__.interactive_heatmap(start_row=0, window_size=100)>

## Diffent Frequency wise cross correlation value

In [19]:
def monthly_heatmap(month):
    # Filter the DataFrame for the selected month
    df_god_tmp = god_df.loc[god_df.index.month == month]
    
    correlation_matrix = df_god_tmp.corr()
    
    # Plot the heatmap
    plt.figure(figsize=(15, 15))
    sns.heatmap(correlation_matrix, annot=True, vmin=-1.0, vmax=1.0, cmap=cmap, center=0)
    plt.title(f'Correlation Matrix for Month {month}')
    plt.show()

# Create a slider for month selection
month_slider = IntSlider(value=1, min=1, max=12, step=1, description='Month:', continuous_update=False)

# Use interact to create an interactive widget
interact(monthly_heatmap, month=month_slider)

interactive(children=(IntSlider(value=1, continuous_update=False, description='Month:', max=12, min=1), Output…

<function __main__.monthly_heatmap(month)>

In [20]:
def quarter_heatmap(quarter):
    # Filter the DataFrame for the selected month
    df_god_tmp = god_df.loc[god_df.index.quarter == quarter]
    
    correlation_matrix = df_god_tmp.corr()
    
    # Plot the heatmap
    plt.figure(figsize=(15, 15))
    sns.heatmap(correlation_matrix, annot=True, vmin=-1.0, vmax=1.0, cmap=cmap, center=0)
    plt.title(f'Correlation Matrix for quarter {quarter}')
    plt.show()

# Create a slider for month selection
quarter_slider = IntSlider(value=1, min=1, max=4, step=1, description='quarter:', continuous_update=True)

# Use interact to create an interactive widget
interact(quarter_heatmap, quarter=quarter_slider)

interactive(children=(IntSlider(value=1, description='quarter:', max=4, min=1), Output()), _dom_classes=('widg…

<function __main__.quarter_heatmap(quarter)>

In [21]:
def week_heatmap(week):
    # Filter the DataFrame for the selected month
    df_god_tmp = god_df.loc[god_df.index.isocalendar().week == week]
    
    # Calculate the correlation matrix
    correlation_matrix = df_god_tmp.corr()
    
    # Plot the heatmap
    plt.figure(figsize=(15, 15))
    sns.heatmap(correlation_matrix, annot=True, vmin=-1.0, vmax=1.0, cmap=cmap, center=0)
    plt.title(f'Correlation Matrix for week {week}')
    plt.show()

# Create a slider for month selection
weekly_slider = IntSlider(value=1, min=1, max=52, step=1, description='Week:', continuous_update=True)

# Use interact to create an interactive widget
interact(week_heatmap, week=weekly_slider)

interactive(children=(IntSlider(value=1, description='Week:', max=52, min=1), Output()), _dom_classes=('widget…

<function __main__.week_heatmap(week)>

In [22]:
sns.heatmap(god_df.corr(), cmap=cmap, vmin=-1, vmax=1.0, annot=True)

<Axes: title={'center': 'Rank at hourly level'}>

# Country analysis